# amsc_demo — AMSC Genie Demo 沙盒建置 / 拆除（測試用，一次性）

- 用途：把 Volume 上的 AMSC genie demo pack（CSV + YAML + SQL）建成 Delta 表、view、table function，給 Genie Space「AMSC 車用顯示市場（Omdia 2Q26）」用。**不是正式 MI DataHub 管線**，不排程。
- 規劃：`PM_Head/.../amsc_demo/AMSC_Databricks_Genie_Demo_Plan.md`（v0.2，§4 notebook 規格）
- 輸入：`{pack_dir}/manifest.yaml`、`data/*.csv`、`genie_space/amsc_views.sql`、`genie_space/genie_space.yaml`
- 輸出（全部 `amsc_` 開頭，在共用 dev schema）：
  - 表：`amsc_fact_app_quarter`、`amsc_fact_supply_chain`、`amsc_fact_report_kpi`、`amsc_dim_application`、`amsc_dim_panel_maker`、`amsc_dim_client`
  - view：`amsc_v_app_annual`、`amsc_v_maker_share`、`amsc_v_maker_client_share`；function：`amsc_fn_client_supplier_mix`
- 表名、欄位、型別、COMMENT、檢核值都**從 `manifest.yaml` 讀**，notebook 不寫死；pack 更新後重跑即可（全量覆寫）。
- 拆除：`teardown=true` 跑一次，只刪 `amsc_` 開頭的物件。**禁止 `DROP SCHEMA`**（`mi3_datahub_dev` 是共用 schema）。

> ⚠️ Omdia 授權付費資料：pack 不進 git；表只 GRANT 給 Demo 群組（`grant_group`），存取控制靠 UC GRANT，不靠 prompt。

**執行前**：把 `genie_demo_pack/` 的**內容**（不含 `_build/`）上傳到 `pack_dir`，使 `{pack_dir}/manifest.yaml` 存在。
在 Catalog Explorer 拖整個資料夾上傳會多一層 `genie_demo_pack/`，這時 `pack_dir` 要加上 `/genie_demo_pack`。

In [ ]:
# [c01] params
# Demo 沙盒，預設值取自 demo 規劃 §4（不是 config/project.yml 的 prod schema）。
# 注意：widget 建立過後改預設值不會生效；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。
import re

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_dev")
dbutils.widgets.text("pack_dir", "/Volumes/micenter/mi3_datahub_dev/test/amsc_demo")
dbutils.widgets.dropdown("teardown", "false", ["false", "true"])  # true = 只拆除 amsc_* 物件，不建表
dbutils.widgets.text("grant_group", "")                          # 空 = 不 GRANT；填 Demo 群組名
dbutils.widgets.text("sample_qids", "Q02,Q06,Q08")              # 建完後抽跑的 genie_space.yaml 範例題

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
pack_dir = dbutils.widgets.get("pack_dir").strip().rstrip("/")
teardown = dbutils.widgets.get("teardown") == "true"
grant_group = dbutils.widgets.get("grant_group").strip()
sample_qids = [q.strip() for q in dbutils.widgets.get("sample_qids").split(",") if q.strip()]

# catalog / schema 會直接拼進 SQL，只允許一般識別字
assert re.fullmatch(r"\w+", catalog) and re.fullmatch(r"\w+", schema), "catalog / schema 只能是英數字與底線"
assert pack_dir.startswith("/Volumes/"), "pack_dir 必須是 UC Volume 路徑（/Volumes/...）"
print(f"catalog={catalog} schema={schema} pack_dir={pack_dir} teardown={teardown} grant_group={grant_group or '-'}")

In [ ]:
# [c02] imports
import os
import re

from pyspark.sql import types as T

PREFIX = "amsc_"  # 共用 dev schema：只動這個前綴的物件
TYPE_MAP = {
    "STRING": T.StringType(),
    "INT": T.IntegerType(),
    "DOUBLE": T.DoubleType(),
    "BOOLEAN": T.BooleanType(),
    "DATE": T.DateType(),
}

In [ ]:
# [c03] helpers
# 純函式（無 spark / dbutils / I/O），tests/test_amsc_demo.py 會載入測試。


def sql_str(s: str) -> str:
    """轉成 Spark SQL 字串常值：反斜線與單引號跳脫。"""
    return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"


def spark_schema(columns: list[dict]) -> T.StructType:
    """manifest 的 columns → StructType（全部 nullable；CSV 空字串 = NULL）。"""
    unknown = sorted({c["type"] for c in columns} - TYPE_MAP.keys())
    if unknown:
        raise ValueError(f"manifest 有未支援的型別：{unknown}")
    return T.StructType([T.StructField(c["name"], TYPE_MAP[c["type"]], True) for c in columns])


def ddl_columns(columns: list[dict]) -> str:
    """manifest 的 columns → CREATE TABLE 欄位定義（含 COMMENT）。"""
    parts = []
    for c in columns:
        col = f"`{c['name']}` {c['type']}"
        if c.get("comment"):
            col += f" COMMENT {sql_str(c['comment'])}"
        parts.append(col)
    return ",\n  ".join(parts)


def tblproperties(props: dict) -> str:
    return ", ".join(f"{sql_str(k)} = {sql_str(v)}" for k, v in props.items())


def split_sql(text: str) -> list[str]:
    """以「; 換行」切 statement，丟掉只有註解 / 空白的片段。"""
    out = []
    for piece in re.split(r";[ \t]*(?:\r?\n|$)", text):
        body = [ln for ln in piece.splitlines() if ln.strip() and not ln.strip().startswith("--")]
        if body:
            out.append(piece.strip())
    return out


def retarget_sql(sql: str, src_ns: str, dst_ns: str) -> str:
    """pack 的 SQL 寫死 manifest target（如 micenter.mi3_datahub_dev）；widget 改了就替換。"""
    return sql if src_ns == dst_ns else sql.replace(f"{src_ns}.", f"{dst_ns}.")


def created_object(sql: str) -> str:
    """取出 CREATE ... VIEW/FUNCTION/TABLE 的物件名，給 log 用。"""
    m = re.search(r"CREATE\s+(?:OR\s+REPLACE\s+)?(VIEW|FUNCTION|TABLE)\s+([\w.`]+)", sql, re.I)
    return f"{m.group(1).upper()} {m.group(2)}" if m else sql.splitlines()[0][:80]


def check_ok(actual, expect, tol: float = 1e-6) -> bool:
    if actual is None:
        return False
    return abs(float(actual) - float(expect)) <= tol


def drop_statements(objects: list[tuple[str, str]], ns: str, prefix: str = PREFIX) -> list[str]:
    """(name, kind) → DROP 指令；kind = FUNCTION / VIEW / TABLE。
    順序：function → view → table（function 依賴 view，view 依賴 table）。不在前綴內的物件直接報錯。"""
    order = {"FUNCTION": 0, "VIEW": 1, "TABLE": 2}
    bad = [n for n, _ in objects if not n.startswith(prefix)]
    if bad:
        raise ValueError(f"拒絕刪除非 {prefix} 物件：{bad}")
    return [f"DROP {k} IF EXISTS {ns}.{n}" for n, k in sorted(objects, key=lambda o: (order[o[1]], o[0]))]

In [ ]:
# [c04] teardown
# teardown=true：列出並刪除 {catalog}.{schema} 內所有 amsc_ 開頭的 table / view / function，然後結束 notebook。
# 不讀 pack（Volume 可能已刪）。禁止 DROP SCHEMA。
if teardown:
    ns = f"{catalog}.{schema}"
    tbls = spark.sql(f"""
        SELECT table_name, table_type FROM {catalog}.information_schema.tables
        WHERE table_schema = {sql_str(schema)} AND startswith(table_name, {sql_str(PREFIX)})
    """).collect()
    fns = spark.sql(f"""
        SELECT routine_name FROM {catalog}.information_schema.routines
        WHERE routine_schema = {sql_str(schema)} AND startswith(routine_name, {sql_str(PREFIX)})
    """).collect()
    objects = [(r.table_name, "VIEW" if r.table_type == "VIEW" else "TABLE") for r in tbls]
    objects += [(r.routine_name, "FUNCTION") for r in fns]

    stmts = drop_statements(objects, ns)
    print(f"待刪 {len(stmts)} 個物件：")
    for s in stmts:
        print("  " + s)
    for s in stmts:
        spark.sql(s)
    print("拆除完成。其餘請手動處理：Volume 資料夾 pack_dir、Genie Space、myAgent 的 Genie tool / Knowledge 掛載。")
    dbutils.notebook.exit(f"teardown: dropped {len(stmts)} objects")

In [ ]:
# [c05] load_manifest
try:
    import yaml  # DBR 17.3 / serverless 內建 PyYAML；若缺，請加到 cluster library / serverless environment
except ImportError as e:
    raise ImportError("缺 pyyaml：請在 cluster library（或 serverless environment）加入 pyyaml") from e

manifest_path = f"{pack_dir}/manifest.yaml"
if not os.path.exists(manifest_path):
    nested = f"{pack_dir}/genie_demo_pack/manifest.yaml"
    hint = f"；但 {nested} 存在，請把 pack_dir 改成 {pack_dir}/genie_demo_pack" if os.path.exists(nested) else ""
    raise FileNotFoundError(f"找不到 {manifest_path}{hint}")

with open(manifest_path, encoding="utf-8") as f:
    manifest = yaml.safe_load(f)

target = manifest["target"]
src_ns = f"{target['catalog']}.{target['schema']}"  # pack 內 SQL 寫死的 namespace
dst_ns = f"{catalog}.{schema}"
if src_ns != dst_ns:
    print(f"⚠️ widget 與 manifest target 不一致：以 widget 為準 {dst_ns}（manifest = {src_ns}），SQL 會替換 namespace")
assert target["table_prefix"] == PREFIX, f"manifest table_prefix={target['table_prefix']}，notebook 只處理 {PREFIX}"
assert all(t["name"].startswith(PREFIX) for t in manifest["tables"]), f"manifest 有不以 {PREFIX} 開頭的表"

files = [t["file"] for t in manifest["tables"]] + [manifest["views_sql"], manifest["genie_space_config"]]
missing = [p for p in files if not os.path.exists(f"{pack_dir}/{p}")]
assert not missing, f"pack 缺檔：{missing}"

print(f"pack_version={manifest['pack_version']}  source={manifest['source']}")
for t in manifest["tables"]:
    print(f"  {t['name']:<28} rows={t['rows']:>5}  cols={len(t['columns']):>2}  {t['file']}")

In [ ]:
# [c20] write_tables
# 寫入策略：CREATE OR REPLACE TABLE（帶表 / 欄 COMMENT 與 TBLPROPERTIES）+ insertInto overwrite = 每次全量重建。
# 用 DDL 一次帶齊 metadata，免去逐欄 ALTER COLUMN（幾十個 commit）；REPLACE 保留 UC 權限。
# 讀 CSV：enforceSchema=False 讓 header 欄名必須和 manifest 一致；FAILFAST 遇到轉型失敗直接報錯，不默默變 NULL。
props_sql = tblproperties(target.get("table_properties", {}))

for t in manifest["tables"]:
    fq = f"{dst_ns}.{t['name']}"
    df = (
        spark.read.option("header", True)
        .option("enforceSchema", False)
        .option("mode", "FAILFAST")
        .option("encoding", "UTF-8")
        .option("dateFormat", "yyyy-MM-dd")
        .schema(spark_schema(t["columns"]))
        .csv(f"{pack_dir}/{t['file']}")
    )
    n = df.count()  # demo 小表（< 5k 列），直接數
    if n != t["rows"]:
        raise ValueError(f"{t['name']}：CSV {n} 列，manifest 預期 {t['rows']} 列")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {fq} (
          {ddl_columns(t['columns'])}
        )
        USING DELTA
        COMMENT {sql_str(t['comment'])}
        TBLPROPERTIES ({props_sql})
    """)
    df.write.insertInto(fq, overwrite=True)
    print(f"✅ {fq}  rows={n}")

In [ ]:
# [c21] create_views
# genie_space/amsc_views.sql：3 個 view + 1 個 table function；namespace 依 widget 替換。
with open(f"{pack_dir}/{manifest['views_sql']}", encoding="utf-8") as f:
    view_stmts = [retarget_sql(s, src_ns, dst_ns) for s in split_sql(f.read())]

for s in view_stmts:
    spark.sql(s)
    print(f"✅ {created_object(s)}")

In [ ]:
# [c30] check_expected
# 跑 manifest.expected_checks（SQL 用不帶 namespace 的表名，先切到目標 schema）。任一 FAIL 就 raise。
spark.catalog.setCurrentCatalog(catalog)
spark.catalog.setCurrentDatabase(schema)

results = []
for c in manifest["expected_checks"]:
    actual = spark.sql(c["sql"]).first()[0]
    results.append((c["name"], actual, c["expect"], check_ok(actual, c["expect"])))

print(f"{'check':<20} {'actual':>12} {'expect':>12}  ok")
for name, actual, expect, ok in results:
    print(f"{name:<20} {str(actual):>12} {str(expect):>12}  {'OK' if ok else 'FAIL'}")
failed = [r[0] for r in results if not r[3]]
assert not failed, f"檢核失敗：{failed}"
print(f"全部 {len(results)} 項 OK")

In [ ]:
# [c31] check_sample_questions
# 抽跑 genie_space.yaml 的範例題，印出結果和 expect，供人工比對（不自動判定）。
with open(f"{pack_dir}/{manifest['genie_space_config']}", encoding="utf-8") as f:
    genie_cfg = yaml.safe_load(f)
questions = {q["id"]: q for q in genie_cfg["sample_questions"]}

unknown = [q for q in sample_qids if q not in questions]
assert not unknown, f"genie_space.yaml 沒有這些題號：{unknown}"
for qid in sample_qids:
    q = questions[qid]
    print(f"\n=== {qid}：{q['question']}\n預期：{q.get('expect', '-')}")
    spark.sql(retarget_sql(q["sql"], src_ns, dst_ns)).show(20, truncate=False)

In [ ]:
# [c40] grant
# grant_group 有填才執行。表 / view 給 SELECT、function 給 EXECUTE。
# 群組另外需要 USE CATALOG / USE SCHEMA（共用 schema，請平台 owner 授予，本 notebook 不動）。
if grant_group:
    objs = [(t["name"], "TABLE") for t in manifest["tables"]]
    objs += [tuple(created_object(s).split(" ", 1)[::-1]) for s in view_stmts]  # (fq_name, VIEW/FUNCTION)
    for name, kind in objs:
        fq = name if "." in name else f"{dst_ns}.{name}"
        priv, on = ("EXECUTE", "FUNCTION") if kind == "FUNCTION" else ("SELECT", "TABLE")
        spark.sql(f"GRANT {priv} ON {on} {fq} TO `{grant_group}`")
        print(f"GRANT {priv} ON {on} {fq} TO `{grant_group}`")
else:
    print("grant_group 未填，略過 GRANT")

## 之後的手動步驟

**驗收**：[c30] 全部 OK；SQL editor 執行 `SELECT * FROM micenter.mi3_datahub_dev.amsc_fn_client_supplier_mix('BHTC', 2025)` 有結果。

**Genie Space**（照 pack 的 `genie_space/genie_space.yaml` 填）
1. 名稱：AMSC 車用顯示市場（Omdia 2Q26）；Warehouse：Serverless SQL 2X-Small。
2. 加入 6 張表、3 個 view、1 個 function（yaml 的 `tables` / `functions`）。
3. General Instructions：貼 yaml 的 `general_instructions`；Joins：照 yaml 的 6 組。
4. Sample questions：13 題附 SQL；benchmark 13 題 + 2 題陷阱題，正確率 ≥ 80% 才接 myAgent。答錯優先補 Instructions 或改成 function。
5. 分享：只給 Demo 群組。

**myAgent**
1. Genie tool 掛上述 Space（負責所有數字）。
2. Knowledge 上傳 pack 的 `knowledge/*.md`（負責「為什麼」，引用「報告名 Slide N」）。
3. Prompt：回答格式與路由規則見規劃 §6.2、§6.3。

**拆除**：`teardown=true` 跑本 notebook → 手動刪 Volume `amsc_demo/`、Genie Space、myAgent 掛載。